# 🔄 Incremental Source Data Generator

**SwissLogistics AG, `sl_ingest` incremental feed**

The incremental counterpart to `environment_setup.ipynb`: where setup does a one-time `overwrite` seed, this appends a **fresh batch** to the `sl_ingest` sources on every run and can inject schema drift, bad records, duplicates and late data. Run it before each pipeline execution so the DLT and PySpark tracks always process new, potentially messy data.

**Run-tracking table.** Every run appends to `_generator_runs`, per source: rows added, counts of bad/duplicate/late records, the drift action, the **cumulative active extra columns**, and a `classification`. At the start of each run the notebook rebuilds the persistent drift state from that table, so once a column drifts in it stays in.

Parameters are read like `setup_job.yml` (`env = ${var.env}`), so it drops into a job task unchanged. Generation logic uses plain Python loops + explicit schemas, mirroring the setup notebook, easy to tweak.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import lit, col, row_number
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import random

## Configuration

Row counts come from job parameters (override per run).

### What each dataset gets this run

Two kinds of mess, controlled separately:

1. **Probabilistic anomalies** (`bad`, `dupes`, `late`): turned on per dataset via its `inject` set. When on, they hit a small random share of rows (e.g. ~3% of orders get a missing amount). Left to chance.
2. **Schema drift**: a structural change, so you pick it explicitly per table in `DRIFT_THIS_RUN`. Each listed JSON table gets its next new column this run (deterministic), and that column persists on later runs. Defaults to empty, so drift only happens when you ask for it.
e.g. ["telemetry"] or ["shipment_events", "cdc_records"]. Empty = no drift this run.

Schema drift is JSON-only; the Delta sources (`orders`, `customers`) keep their schema for this scenario and simplicity, so they are never drifted.

In [0]:
# Job parameters arrive via getCurrentBindings(), same pattern as environment_setup.ipynb
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV         = configs.get("env", "dev")
SCHEMA_INGEST = ENV
LANDING_BASE  = f"/Volumes/sl_ingest/{SCHEMA_INGEST}/landing"

# Run-tracking table location
AUDIT_TABLE = f"sl_ingest.{SCHEMA_INGEST}._incremental_generator_runs"

# A new seed every run so each batch differs; BATCH_ID makes appended files/ids unique
BATCH_TS = datetime.now()
BATCH_ID = BATCH_TS.strftime("%Y%m%d%H%M%S")
random.seed(int(BATCH_TS.timestamp()))  # Sets the random number generator seed to the current batch timestamp for reproducible randomness per batch

N_ORDERS    = int(configs.get("n_orders",    "500"))   # new orders this batch
N_CUSTOMERS = int(configs.get("n_customers", "20"))    # customer rows this batch (changes + new)
N_EVENTS    = int(configs.get("n_events",    "5000"))  # new shipment events this batch
N_TELEMETRY = int(configs.get("n_telemetry", "2000"))  # new telemetry readings this batch
N_CDC       = int(configs.get("n_cdc",       "275"))   # orders_cdc rows this batch (updates + inserts)

INJECT_ANOMALIES = configs.get("inject_anomalies", True)
DRIFT_THIS_RUN = configs.get("drift_this_run", "").split(',')


In [0]:
JSON_SOURCES = {"shipment_events", "telemetry", "cdc_records"}

# 1) Probabilistic anomalies per dataset: a small random share of rows when listed.
#    Options: "bad", "dupes", "late".  (Drift is handled separately below.)
DATASETS = {
    "orders":          {"count": N_ORDERS,    "inject": {"bad"}},
    "customers":       {"count": N_CUSTOMERS, "inject": set()},                  # changes + new only
    "shipment_events": {"count": N_EVENTS,    "inject": {"bad", "dupes", "late"}},
    "telemetry":       {"count": N_TELEMETRY, "inject": {"bad"}},
    "cdc_records":     {"count": N_CDC,        "inject": set()},
}

def chance(p):
    """True with probability p (one random draw)."""
    return random.random() < p

def wants(name, anomaly):
    if anomaly == "drift":
        return name in DRIFT_THIS_RUN and name in JSON_SOURCES
    return anomaly in DATASETS[name]["inject"]

print(f"env={ENV}  counts:", {k: v["count"] for k, v in DATASETS.items()})
for k, v in DATASETS.items():
    print(f"  {k:16s} anomalies: {sorted(v['inject']) or 'none'}")
print(f"  drift this run: {DRIFT_THIS_RUN or 'none'}")

In [0]:
spark.sql("USE CATALOG sl_ingest")
print(f"Appending batch {BATCH_ID} to sl_ingest / {LANDING_BASE}")

## Run-tracking table & persistent schema state

The audit table is append-only history. On each run we rebuild `drift_state` = the latest `active_extra_columns` per source, so columns that drifted in on earlier runs keep being emitted. This is what lets the next run "know" a column was permanently added.

In [0]:
# The run-tracking table: append-only history, one row per (batch, source).
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
        batch_id              STRING,
        run_ts                TIMESTAMP,
        env                   STRING,
        source                STRING,
        rows_added            INT,
        bad_records           INT,
        duplicates            INT,
        late_records          INT,
        drift_action          STRING,
        active_extra_columns  STRING,
        classification        STRING
    )
""")

# Rebuild the persistent drift state. For each source, take its most recent audit row
# and read back the columns that have drifted in so far. This is what lets a column that
# appeared on an earlier run keep being emitted this run.
drift_state = {}
if spark.table(AUDIT_TABLE).head(1):                          # table already has rows
    newest_first = Window.partitionBy("source").orderBy(
        col("run_ts").desc(), col("batch_id").desc())
    latest_rows = (
        spark.table(AUDIT_TABLE)
        .withColumn("rn", row_number().over(newest_first))
        .filter("rn = 1")                                     # keep only the newest row per source
        .select("source", "active_extra_columns")
        .collect()
    )
    for r in latest_rows:
        columns = [c for c in (r["active_extra_columns"] or "").split(",") if c]
        if columns:
            drift_state[r["source"]] = columns

print(f"Audit table: {AUDIT_TABLE}")
print(f"Carried-forward drift state: {drift_state or 'none yet'}")


def classify(bad, dup, late, drift_action):
    """Turn the per-source counts into a single label for the audit row."""
    tags = []
    if drift_action and drift_action != "none":
        tags.append("schema_drift")
    if bad > 0:
        tags.append("bad_records")
    if dup > 0:
        tags.append("duplicates")
    if late > 0:
        tags.append("late_data")
    return ",".join(tags) if tags else "healthy"

## Schema-drift helper (deterministic, JSON-only)

When `drift` is enabled for a JSON dataset, each run adds the **next** new candidate column (until the list is exhausted) and re-applies every column added before, so drift accumulates exactly like a real upstream schema change. Delta sources are never drifted.

In [0]:
DRIFT_CANDIDATES = {
    "shipment_events": ["device_firmware", "gps_accuracy_m", "carrier_code"],
    "telemetry":       ["tire_pressure_psi", "battery_voltage", "ambient_temp_c"],
    "cdc_records":     ["source_batch", "ingest_region"],
}

def _drift_value(c):
    if c == "device_firmware":
        return f"fw{random.randint(1, 9)}.{random.randint(0, 9)}"
    if c in ("gps_accuracy_m", "tire_pressure_psi", "battery_voltage", "ambient_temp_c"):
        return str(round(random.uniform(1, 40), 1))
    if c == "source_batch":
        return BATCH_ID
    if c == "ingest_region":
        return random.choice(["CH-N", "CH-S", "CH-E", "CH-W"])
    if c == "carrier_code":
        return random.choice(["SLX", "PSTCH", "DHLCH"])
    return "drift"

def maybe_drift(df, name):
    active = list(drift_state.get(name, []))               # columns added on earlier runs (persist)
    action = "none"
    if wants(name, "drift"):
        nxt = next((c for c in DRIFT_CANDIDATES.get(name, []) if c not in active), None)
        if nxt:
            active.append(nxt)                             # deterministic: add the next new column
            action = f"added column '{nxt}'"
    for c in active:
        df = df.withColumn(c, lit(_drift_value(c)))
    drift_state[name] = active
    return df, action, ",".join(active)

## Reference data

The same lookups used by the initial seed, so generated records stay consistent with the existing dataset.

In [0]:
swiss_cities = ["Zürich", "Bern", "Basel", "Genève", "Lausanne", "Winterthur", "St. Gallen",
                "Luzern", "Biel", "Thun", "Aarau", "Schaffhausen", "Lugano", "Fribourg", "Chur",
                "Neuchâtel", "Zug", "Sion"]
tiers = ["standard", "premium", "enterprise"]
industries = ["pharma", "retail", "manufacturing", "food_beverage", "tech", "finance"]
products = [("PKG_STD", "Standard Parcel", 12.50), ("PKG_EXP", "Express Parcel", 24.90),
            ("PKG_SAME", "Same-Day Delivery", 39.90), ("FRT_PAL", "Pallet Freight", 89.00),
            ("FRT_FTL", "Full Truck Load", 450.00), ("COLD", "Cold Chain Parcel", 34.90),
            ("DOC", "Document Courier", 8.90), ("INTL", "International Shipment", 65.00)]
statuses = ["created", "confirmed", "picked_up", "in_transit", "out_for_delivery",
            "delivered", "failed_delivery", "returned"]
payment_methods = ["invoice", "credit_card", "bank_transfer", "paypal"]
event_types = ["gps_ping", "status_change", "temperature_alert", "delay_alert", "signature_captured"]

## 1 · Orders: incremental append (Delta, CDF)

New orders continuing the existing ID sequence. Bad-record injection adds the wrinkles the silver layer must catch: missing amounts, the occasional negative amount, and a blank destination city.

In [0]:
# Highest existing ORD-###### number, so new ids continue the sequence (0 if empty).
mx = (spark.table(f"sl_ingest.{SCHEMA_INGEST}.orders")
      .selectExpr("max(cast(substring(order_id, 5) as int)) m").collect()[0]["m"]) or 0

order_bad = 0
order_recs = []
for k in range(N_ORDERS):
    oid = f"ORD-{mx + k + 1:06d}"
    cid = random.randint(1, 200)
    prod = random.choice(products)
    qty = random.randint(1, 10) if prod[0].startswith("PKG") else random.randint(1, 3)
    odate = BATCH_TS - timedelta(minutes=random.randint(0, 180))
    status = random.choices(statuses, weights=[20, 15, 10, 15, 10, 20, 5, 5])[0]
    ddate = odate + timedelta(days=random.randint(1, 5)) if status in ("delivered", "returned") else None
    amount = round(prod[2] * qty * random.uniform(0.9, 1.1), 2)
    origin, dest = random.choice(swiss_cities), random.choice(swiss_cities)
    if wants("orders", "bad"):
        if chance(0.03):
            amount = None          # missing amount
            order_bad += 1
        elif chance(0.01):
            amount = -abs(amount)  # negative amount
            order_bad += 1
        if chance(0.01):
            dest = ""              # blank destination
            order_bad += 1
    order_recs.append((oid, cid, prod[0], prod[1], qty, amount, status,
                       random.choice(payment_methods), odate, ddate, origin, dest))

order_schema = StructType([
    StructField("order_id", StringType()), StructField("customer_id", IntegerType()),
    StructField("product_code", StringType()), StructField("product_name", StringType()),
    StructField("quantity", IntegerType()), StructField("total_amount", DoubleType(), True),
    StructField("status", StringType()), StructField("payment_method", StringType()),
    StructField("order_date", TimestampType()), StructField("delivery_date", TimestampType(), True),
    StructField("origin_city", StringType()), StructField("destination_city", StringType())])

(spark.createDataFrame(order_recs, order_schema).write.mode("append")
 .option("delta.enableChangeDataFeed", "true").saveAsTable(f"sl_ingest.{SCHEMA_INGEST}.orders"))
n_orders_added = len(order_recs)
print(f"  ✓ orders: +{n_orders_added} rows (Delta, CDF) | bad: {order_bad}")

## 2 · Customers: real SCD2 changes (Delta, CDF)

This no longer stamps random near-duplicate rows. It reads the **current** customers from `sl_ingest.customers` and uses them as the reference to produce genuine changes:

- **Attribute updates**: a subset of existing customers get a new tier and a new address/city. Their identity (`customer_id`, `company_name`, `contact_email`, `signup_date`) is copied unchanged, only the changing attributes move.
- **Churn (soft-delete)**: a few customers have their current row retired with no replacement, so no current version remains.
- **New customers**: brand-new ids continue the sequence.
- **Unchanged**: everyone not picked is left alone, so silver has to ignore no-ops.

The SCD2 invariant (exactly one `is_current = True` row per customer) is preserved. When a customer changes, the new version is appended (`is_current = True`, `changed_at = null`) and the previous current row is flipped to `is_current = False`, `changed_at = <batch ts>`. Those flips are real Delta `UPDATE`s, so the source Change Data Feed emits `update_preimage` / `update_postimage` for them and `insert` for the appended versions. The bronze loader picks all of that up via `readChangeFeed`.

In [0]:
# ── Customers: real SCD2 changes built from the live source table ──
# sl_ingest.customers is an append-only, versioned SCD2 source: exactly one
# is_current=True row per customer_id, superseded versions are is_current=False
# with changed_at = the time they were retired.
#
# We read the CURRENT customers and use them as the reference to produce genuine
# changes (no random near-duplicates):
#   1. ATTRIBUTE UPDATE   copy identity (id, company_name, contact_email,
#      signup_date), change tier + address/city, append the new current version,
#      and flip the old current row to is_current=False.
#   2. CHURN / SOFT-DELETE flip a few current rows to is_current=False with no
#      replacement, so no current version remains.
#   3. NEW CUSTOMERS      brand-new ids continuing the sequence.
#   4. UNCHANGED          everyone not picked is left untouched this run.
#
# The is_current flips are real Delta UPDATEs, so the source Change Data Feed
# emits update_preimage/update_postimage for them; the appended rows emit inserts.
from delta.tables import DeltaTable

source_tbl = f"sl_ingest.{SCHEMA_INGEST}.customers"
streets = ["Bahnhofstrasse", "Seestrasse", "Industriestrasse", "Hauptstrasse", "Marktplatz", "Dorfstrasse"]

# Live reference: the current version of every customer right now.
current_by_id = {r["customer_id"]: r
                 for r in spark.table(source_tbl).filter("is_current = true").collect()}
current_ids = list(current_by_id.keys())
mxc = max(current_ids) if current_ids else 0

# Activity budget for this batch, split across change types.
n_update = max(1, round(N_CUSTOMERS * 0.50))   # existing customers that change
n_delete = max(0, round(N_CUSTOMERS * 0.15))   # existing customers that churn
n_new    = max(1, round(N_CUSTOMERS * 0.35))   # brand-new customers

random.shuffle(current_ids)
update_ids = current_ids[:n_update]
delete_ids = current_ids[n_update:n_update + n_delete]

# Appended rows: new current versions (for the updated customers) + brand-new customers.
cust_recs = []
for cid in update_ids:
    base = current_by_id[cid]
    new_tier = random.choice([t for t in tiers if t != base["tier"]])
    cust_recs.append((cid, base["company_name"], base["contact_email"],
                      f"{random.randint(1, 200)} {random.choice(streets)}",
                      random.choice(swiss_cities), f"{random.randint(1000, 9999)}",
                      new_tier, base["industry"],
                      base["signup_date"], None, True))            # new current: changed_at null
for j in range(n_new):
    cid = mxc + j + 1
    cust_recs.append((cid, f"Company_{cid:04d}", f"contact_{cid}@company{cid}.ch",
                      f"{random.randint(1, 200)} Hauptstrasse", random.choice(swiss_cities),
                      f"{random.randint(1000, 9999)}", random.choice(tiers), random.choice(industries),
                      BATCH_TS, None, True))

cust_schema = StructType([
    StructField("customer_id", IntegerType()), StructField("company_name", StringType()),
    StructField("contact_email", StringType()), StructField("address", StringType()),
    StructField("city", StringType()), StructField("postal_code", StringType()),
    StructField("tier", StringType()), StructField("industry", StringType()),
    StructField("signup_date", TimestampType()), StructField("changed_at", TimestampType()),
    StructField("is_current", BooleanType())])

# 1) Retire the superseded + churned current rows (real Delta UPDATE -> CDF update events).
flip_ids = update_ids + delete_ids
if flip_ids:
    (DeltaTable.forName(spark, source_tbl)
     .update(condition=(col("is_current")) & (col("customer_id").isin(flip_ids)),
             set={"is_current": lit(False), "changed_at": lit(BATCH_TS)}))

# 2) Append the new current versions + brand-new customers (CDF inserts).
if cust_recs:
    (spark.createDataFrame(cust_recs, cust_schema).write.mode("append")
     .option("delta.enableChangeDataFeed", "true").saveAsTable(source_tbl))

n_cust_added = len(cust_recs)
print(f"  ✓ customers: {len(update_ids)} changed, {len(delete_ids)} churned, {n_new} new "
      f"| appended {n_cust_added} rows, retired {len(flip_ids)} current rows")

## 3 · Shipment events: JSON append

High-volume tracking events written as **new JSON files** (Auto Loader reads them as an incremental batch). When enabled: ~3% duplicate event IDs and ~4% late / out-of-order timestamps; bad records null the vehicle ID or push temperatures out of range. Persistent schema drift may add a column for this and all future batches.

In [0]:
max_order_no = mx + N_ORDERS  # highest order id after this batch's orders were added
events_bad = events_dupes = events_late = 0
event_recs = []
for i in range(N_EVENTS):
    eid = f"EVT-{BATCH_ID}-{i:06d}"
    oid = f"ORD-{random.randint(1, max_order_no):06d}"
    veh = f"VH-{random.randint(1, 200):03d}"
    et = random.choices(event_types, weights=[60, 20, 5, 10, 5])[0]
    ts = BATCH_TS - timedelta(minutes=random.randint(0, 240))
    if wants("shipment_events", "late") and chance(0.04):
        ts = BATCH_TS - timedelta(days=random.randint(3, 30))  # late / out-of-order
        events_late += 1
    lat = round(random.uniform(46.0, 47.8), 6) if et == "gps_ping" else None
    lon = round(random.uniform(6.0, 10.5), 6) if et == "gps_ping" else None
    st = random.choice(statuses) if et == "status_change" else None
    tmp = round(random.gauss(4, 2), 1) if et == "temperature_alert" else None
    if wants("shipment_events", "bad"):
        if chance(0.02):
            veh = None    # missing vehicle id
            events_bad += 1
        if tmp is not None and chance(0.05):
            tmp = 999.0   # out-of-range temperature
            events_bad += 1
    row = (eid, oid, veh, et, ts, lat, lon, st, tmp, random.choice(["kafka", "iot_hub", "api"]))
    event_recs.append(row)
    if wants("shipment_events", "dupes") and chance(0.03):
        event_recs.append(row)  # duplicate event
        events_dupes += 1

event_schema = StructType([
    StructField("event_id", StringType()), StructField("order_id", StringType()),
    StructField("vehicle_id", StringType(), True), StructField("event_type", StringType()),
    StructField("event_timestamp", TimestampType()),
    StructField("latitude", DoubleType(), True), StructField("longitude", DoubleType(), True),
    StructField("shipment_status", StringType(), True), StructField("temperature_celsius", DoubleType(), True),
    StructField("source_system", StringType())])

df_events, drift_e, active_e = maybe_drift(spark.createDataFrame(event_recs, event_schema), "shipment_events")
df_events.repartition(2).write.mode("append").json(f"{LANDING_BASE}/shipment_events")
n_events_added = len(event_recs)
print(f"  ✓ shipment_events: +{n_events_added} json records | bad: {events_bad} dupes: {events_dupes} late: {events_late} | drift: {drift_e}")

## 4 · Vehicle telemetry: JSON append

Sensor readings keeping the same 60%-to-5-vehicles **skew** as the initial seed so partition/clustering optimisation stays relevant. Bad records inject impossible speeds; persistent drift may add a column.

In [0]:
import uuid
tel_bad = 0
tel_recs = []
for i in range(N_TELEMETRY):
    veh = f"VH-{random.randint(1, 5):03d}" if random.random() < 0.6 else f"VH-{random.randint(6, 200):03d}"
    ts = BATCH_TS - timedelta(minutes=random.randint(0, 240), seconds=random.randint(0, 59))
    speed = round(random.uniform(0, 120), 1)
    if wants("telemetry", "bad") and chance(0.01):
        speed = round(random.uniform(300, 900), 1)  # impossible speed
        tel_bad += 1
    fuel = round(random.uniform(0, 100), 1)
    etemp = round(random.gauss(22, 8), 1)
    ctemp = round(random.gauss(4, 3), 1) if random.random() < 0.3 else None
    odo = random.randint(0, 500000)
    tel_recs.append((str(uuid.uuid4()), veh, ts, speed, fuel, etemp, ctemp, odo,
                     random.choice(["idle", "moving", "loading", "maintenance"])))

tel_schema = StructType([
    StructField("reading_id", StringType()),
    StructField("vehicle_id", StringType()), StructField("reading_timestamp", TimestampType()),
    StructField("speed_kmh", DoubleType()), StructField("fuel_pct", DoubleType()),
    StructField("engine_temp_c", DoubleType()), StructField("cargo_temp_c", DoubleType(), True),
    StructField("odometer_km", IntegerType()), StructField("vehicle_status", StringType())])

df_tel, drift_t, active_t = maybe_drift(spark.createDataFrame(tel_recs, tel_schema), "telemetry")
df_tel.repartition(1).write.mode("append").json(f"{LANDING_BASE}/telemetry")
n_tel_added = len(tel_recs)
print(f"  ✓ telemetry: +{n_tel_added} json records | bad: {tel_bad} | drift: {drift_t}")

## 5 · Orders CDC: daily change batch (JSON append)

A new daily change feed: updates to existing orders plus a few inserts, for `MERGE` / `APPLY CHANGES INTO` upsert handling. Sized relative to the order batch.

In [0]:
max_order_no = mx + N_ORDERS  # highest order id after this batch's orders were added
n_cdc_insert = max(1, round(N_CDC * 0.27))
n_cdc_update = max(1, N_CDC - n_cdc_insert)
cdc_recs = []
for _ in range(n_cdc_update):     # status updates to existing orders
    oid = f"ORD-{random.randint(1, max_order_no):06d}"
    cdc_recs.append((oid, None, None, None, None, None,
                     random.choice(["delivered", "failed_delivery", "returned"]),
                     None, None, BATCH_TS, None, None, "UPDATE"))
for j in range(n_cdc_insert):    # brand-new orders arriving via CDC
    oid = f"ORD-{max_order_no + j + 1:06d}"
    prod = random.choice(products)
    qty = random.randint(1, 5)
    cdc_recs.append((oid, random.randint(1, 200), prod[0], prod[1], qty, round(prod[2] * qty, 2),
                     "created", random.choice(payment_methods), BATCH_TS, None,
                     random.choice(swiss_cities), random.choice(swiss_cities), "INSERT"))

cdc_schema = StructType([
    StructField("order_id", StringType()), StructField("customer_id", IntegerType(), True),
    StructField("product_code", StringType(), True), StructField("product_name", StringType(), True),
    StructField("quantity", IntegerType(), True), StructField("total_amount", DoubleType(), True),
    StructField("status", StringType()), StructField("payment_method", StringType(), True),
    StructField("order_date", TimestampType(), True), StructField("delivery_date", TimestampType(), True),
    StructField("origin_city", StringType(), True), StructField("destination_city", StringType(), True),
    StructField("_change_type", StringType())])

df_cdc, drift_c, active_c = maybe_drift(spark.createDataFrame(cdc_recs, cdc_schema), "cdc_records")
df_cdc.coalesce(1).write.mode("append").json(f"{LANDING_BASE}/cdc_records")
n_cdc_added = len(cdc_recs)
print(f"  ✓ cdc_records: +{n_cdc_added} json records | drift: {drift_c}")

## 6 · Malformed lines (optional)

`df.write.json` only ever emits valid JSON, so to exercise `badRecordsPath` / `_rescued_data` / `PERMISSIVE` parsing we drop a small file of genuinely broken lines into the `shipment_events` landing dir. These count toward `shipment_events` bad records in the audit. Only runs when bad-record injection is on.

In [0]:
malformed_n = 0
if wants("shipment_events", "bad"):
    broken_lines = [
        '{"event_id":"EVT-BROKEN-1","order_id":"ORD-000001","vehicle_id":"VH-007"',           # truncated, no closing brace
        '{"event_id":"EVT-BROKEN-2","event_type":"gps_ping","temperature_celsius":"NaN-ish"}',  # wrong type
        'this is not json at all',                                                             # raw garbage
        '',                                                                                    # blank line
    ]
    malformed_path = f"{LANDING_BASE}/shipment_events/_malformed_{BATCH_ID}.json"
    dbutils.fs.put(malformed_path, "\n".join(broken_lines) + "\n", overwrite=True)
    malformed_n = len(broken_lines)
    events_bad += malformed_n
    print(f"  ✓ shipment_events: +{malformed_n} malformed lines -> {malformed_path}")
else:
    print("  (skipped malformed lines: 'bad' not enabled for shipment_events)")

## Write the run-tracking rows

One row per source for this batch, counts, drift action, the cumulative active columns, and the derived classification, appended to the audit table.

In [0]:
audit_rows = [
    (BATCH_ID, BATCH_TS, ENV, "orders",          n_orders_added, order_bad,  0,            0,           "none",  "",       classify(order_bad, 0, 0, "none")),
    (BATCH_ID, BATCH_TS, ENV, "customers",        n_cust_added,   0,          0,            0,           "none",  "",       classify(0, 0, 0, "none")),
    (BATCH_ID, BATCH_TS, ENV, "shipment_events",  n_events_added, events_bad, events_dupes, events_late, drift_e, active_e, classify(events_bad, events_dupes, events_late, drift_e)),
    (BATCH_ID, BATCH_TS, ENV, "telemetry",        n_tel_added,    tel_bad,    0,            0,           drift_t, active_t, classify(tel_bad, 0, 0, drift_t)),
    (BATCH_ID, BATCH_TS, ENV, "cdc_records",      n_cdc_added,    0,          0,            0,           drift_c, active_c, classify(0, 0, 0, drift_c)),
]

audit_schema = StructType([
    StructField("batch_id", StringType()), StructField("run_ts", TimestampType()),
    StructField("env", StringType()), StructField("source", StringType()),
    StructField("rows_added", IntegerType()), StructField("bad_records", IntegerType()),
    StructField("duplicates", IntegerType()), StructField("late_records", IntegerType()),
    StructField("drift_action", StringType()), StructField("active_extra_columns", StringType()),
    StructField("classification", StringType())])

spark.createDataFrame(audit_rows, audit_schema).write.mode("append").saveAsTable(AUDIT_TABLE)
print(f"  ✓ {AUDIT_TABLE}: +{len(audit_rows)} rows for batch {BATCH_ID}")
for r in audit_rows:
    print(f"    {r[3]:16s} rows+{r[4]:<6} drift={r[8]:24s} class={r[10]}")

## Summary

In [0]:
print("=" * 64)
print(f"SwissLogistics, incremental batch {BATCH_ID} complete  (env={ENV})")
print("=" * 64)
for name in ["shipment_events", "telemetry", "cdc_records"]:
    n_files = len(dbutils.fs.ls(f"{LANDING_BASE}/{name}"))
    print(f"  {LANDING_BASE}/{name}: {n_files} files total")
for t in ["orders", "customers"]:
    total = spark.table(f"sl_ingest.{SCHEMA_INGEST}.{t}").count()
    print(f"  sl_ingest.{SCHEMA_INGEST}.{t}: {total:,} rows total")
print()
print("This batch in the tracking table:")
display(spark.sql(f"SELECT source, rows_added, bad_records, duplicates, late_records, drift_action, active_extra_columns, classification FROM {AUDIT_TABLE} WHERE batch_id = '{BATCH_ID}' ORDER BY source"))